# Gravity 2D — Paper 1 Table 10 Reproduction (CIFAR-10)

This notebook reproduces the small-scale 2D image classification result from:

> Chiang, C.-W. (2026). *Gravity: A Physics-Inspired O(N) Framework with O(1) Streaming State Across Dimensions*. Zenodo.

**Target**: Test accuracy ≈ **82.3%** on CIFAR-10, using 4-direction parallel scans on the 2D patch grid.

## Configuration

| Component | Value |
|---|---|
| Architecture | Gravity-2D (4-direction scan: → ← ↓ ↑) |
| d_model | 128 |
| K (field parameters) | 15 |
| S (scales) | 3 |
| R (local window radius) | 1 (3×3 spatial window) |
| Layers | 4 |
| Patch size | 4×4 |
| Patch grid | 8×8 = 64 patches |
| Image resolution | 32×32 RGB |
| Classes | 10 |
| Batch size | 128 |
| Optimizer | AdamW, lr=2e-3, weight decay=0.05 |
| Epochs | 50 |
| Mixed precision | AMP (cuda) |
| Hardware | NVIDIA A100 40GB |
| Random seed | 42 |
| Expected params | ~653K |
| **Expected test accuracy** | **82.3%** |

## Why 2D matters

The Gravity field equation is natively multi-dimensional. The 1D scan generalises to 2D
by performing **four independent parallel scans** along the rows (→ ←) and columns (↓ ↑),
then averaging the resulting potential fields. This yields rotation-symmetric spatial
context with O(N) compute, where N is the number of patches.

This is the same architecture as the 1D version — only the field solver changes from
1 scan to 4. Density bottleneck, local window attention, and physics feature extraction
are identical to the 1D case.

## Reproducibility

Single-seed result with seed=42. Per Paper 1 Section 4.8, Gravity-2D achieves 82.3%
which is +2.0% over ViT-Tiny (80.3%) and within 7.4% of a translation-equivariant CNN
(89.7%) — using the density bottleneck as a physics-derived spatial inductive bias.

## License

AGPL-3.0. Patent notice: implements technology covered by pending U.S. patent
application (March 2026). Commercial inquiries: chiangjw90@gmail.com


In [1]:
# === Environment setup ===
import os, math, random, time, gc, hashlib, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = torch.cuda.is_available()
print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")
print(f"Mixed precision (AMP): {use_amp}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


PyTorch: 2.10.0+cu128
Device: cuda
Mixed precision (AMP): True
GPU: NVIDIA A100-SXM4-40GB


## 1. Data: CIFAR-10 with standard augmentation

CIFAR-10 is a public benchmark of 60,000 32×32 RGB images across 10 classes
(50K train, 10K test). We use the standard training-time augmentation
(random crop + horizontal flip) and normalisation.


In [2]:
# === Load CIFAR-10 with standard augmentation ===
from torchvision import datasets, transforms

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)
BATCH_SIZE = 128

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_ds = datasets.CIFAR10('./data', train=True,  download=True, transform=train_transform)
test_ds  = datasets.CIFAR10('./data', train=False, download=True, transform=test_transform)

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=True, num_workers=2, generator=g)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train images: {len(train_ds):,}")
print(f"Test images:  {len(test_ds):,}")
print(f"Train batches: {len(train_loader)}")
print(f"Test batches:  {len(test_loader)}")


100%|██████████| 170M/170M [00:13<00:00, 13.0MB/s]


Train images: 50,000
Test images:  10,000
Train batches: 390
Test batches:  79


## 2. Gravity-2D Architecture

A Gravity-2D block consists of:

1. **Coin projection** — Linear(d_model → K), then π·tanh bounding
2. **Density bottleneck** — ρ(i,j) = Σₖ wₖ · aₖ(i,j)² (single-channel, C=1)
3. **4-direction parallel field solver** — independent scans → ← ↓ ↑ on the 2D grid;
   each scan uses its own learnable λₛ and βₛ per scale s
4. **Local window attention** — 3×3 spatial window (R=1); scores combine
   field potential differences and coin similarities
5. **Physics feature extraction** — composite feature vector → MLP → residual
6. **FFN** — standard FFN residual

The 4-direction scan provides rotation-symmetric spatial context — averaging
the four directional fields yields an isotropic potential that captures
information from all directions in O(N) compute.


In [3]:
# === 2D Field Solver: 4-Direction Parallel Scan ===

class FieldSolver2D(nn.Module):
    """Multi-scale EMA on scalar density via 4 independent directional scans."""
    def __init__(self, n_scales=3):
        super().__init__()
        self.n_scales = n_scales
        # Each of 4 directions gets its own learnable lambda and beta per scale
        self.log_lambdas = nn.Parameter(
            torch.linspace(0.7, 4.9, n_scales).unsqueeze(0).expand(4, -1).clone())
        self.log_betas   = nn.Parameter(
            torch.linspace(-0.5, 0.5, n_scales).unsqueeze(0).expand(4, -1).clone())

    def _scan_1d(self, rho_1d, dir_idx):
        """Single-direction parallel scan along the last dim."""
        B, L = rho_1d.shape
        S = self.n_scales
        device = rho_1d.device

        lambdas = torch.exp(self.log_lambdas[dir_idx]).clamp(1.0, 1000.0)
        alphas  = torch.exp(-1.0 / lambdas)
        betas   = torch.exp(self.log_betas[dir_idx]).clamp(0.01, 100.0)

        b = rho_1d.unsqueeze(1) * betas.view(1, -1, 1)            # [B, S, L]
        a = alphas.view(1, S, 1).expand(B, S, L).clone()           # [B, S, L]

        # Blelloch-style parallel scan via iterative doubling
        stride = 1
        while stride < L:
            b = a * F.pad(b, (stride, 0))[:, :, :L] + b
            a = a * F.pad(a, (stride, 0), value=1.0)[:, :, :L]
            stride *= 2

        phi = b

        # Causal running-mean removal (de-mean the field)
        cumsum = phi.cumsum(-1)
        counts = torch.arange(1, L + 1, device=device, dtype=phi.dtype)
        phi = (phi - cumsum / counts).clamp(-1000, 1000)

        # Causal gradient
        grad_phi = torch.zeros_like(phi)
        grad_phi[:, :, 0]  = phi[:, :, 0]
        grad_phi[:, :, 1:] = phi[:, :, 1:] - phi[:, :, :-1]

        return phi, grad_phi

    def forward(self, rho_2d):
        """
        rho_2d: [B, H, W] scalar density per grid position
        Returns:
            phi:      [B, 4, S, H, W]  potential fields for 4 directions
            grad_phi: [B, 4, S, H, W]  gradient fields
        """
        B, H, W = rho_2d.shape
        S = self.n_scales
        all_phi, all_gp = [], []

        # → rightward (along W, forward)
        rr = rho_2d.reshape(B * H, W)
        p, g = self._scan_1d(rr, 0)
        all_phi.append(p.reshape(B, H, S, W).permute(0, 2, 1, 3))
        all_gp.append(g.reshape(B, H, S, W).permute(0, 2, 1, 3))

        # ← leftward (along W, reversed)
        p, g = self._scan_1d(rr.flip(-1), 1)
        all_phi.append(p.flip(-1).reshape(B, H, S, W).permute(0, 2, 1, 3))
        all_gp.append(g.flip(-1).reshape(B, H, S, W).permute(0, 2, 1, 3))

        # ↓ downward (along H, forward)
        rc = rho_2d.permute(0, 2, 1).reshape(B * W, H)
        p, g = self._scan_1d(rc, 2)
        all_phi.append(p.reshape(B, W, S, H).permute(0, 2, 3, 1))
        all_gp.append(g.reshape(B, W, S, H).permute(0, 2, 3, 1))

        # ↑ upward (along H, reversed)
        p, g = self._scan_1d(rc.flip(-1), 3)
        all_phi.append(p.flip(-1).reshape(B, W, S, H).permute(0, 2, 3, 1))
        all_gp.append(g.flip(-1).reshape(B, W, S, H).permute(0, 2, 3, 1))

        return torch.stack(all_phi, dim=1), torch.stack(all_gp, dim=1)


In [4]:
# === 2D Local Window Attention ===

class LocalAttention2D(nn.Module):
    """Causal local 2D window attention with field-derived scoring."""
    def __init__(self, n_gen, n_scales=3, radius=1):
        super().__init__()
        self.n_scales = n_scales
        self.radius = radius
        self.W = (2 * radius + 1) ** 2          # window contains (2R+1)^2 positions

        # Scoring weights
        self.raw_w_phi  = nn.Parameter(torch.tensor(0.5))
        self.raw_w_coin = nn.Parameter(torch.tensor(0.5))
        self.log_sigma2 = nn.Parameter(torch.tensor(0.0))

        # Single-channel density projection: K -> 1
        self.density_weights = nn.Parameter(torch.ones(n_gen) / n_gen)

    def compute_density(self, coins):
        """coins [B,H,W,K] -> density [B,H,W] scalar per position."""
        return (coins ** 2) @ F.softplus(self.density_weights)

    def forward(self, coins, phi):
        """
        coins: [B, H, W, K]
        phi:   [B, 4, S, H, W]
        Returns:
            attn: [B, H, W, win]
            neighbor_features: [B, H, W, K]
        """
        B, H, Wg, K = coins.shape
        S = self.n_scales
        R = self.radius

        w_phi  = F.softplus(self.raw_w_phi)
        w_coin = F.softplus(self.raw_w_coin)
        sigma2 = torch.exp(self.log_sigma2).clamp(0.01, 100.0)

        # Aggregate field over the 4 directions
        phi_agg = phi.mean(dim=1)                          # [B, S, H, W]

        # Unfold a (2R+1)x(2R+1) window at each position with zero padding
        cp = F.pad(coins.permute(0, 3, 1, 2), (R, R, R, R)) \
                .unfold(2, 2*R+1, 1).unfold(3, 2*R+1, 1)    # [B, K, H, W, w, w]
        cw = cp.permute(0, 2, 3, 4, 5, 1).reshape(B, H, Wg, self.W, K)

        pp = F.pad(phi_agg, (R, R, R, R)) \
                .unfold(2, 2*R+1, 1).unfold(3, 2*R+1, 1) \
                .reshape(B, S, H, Wg, self.W)

        # Coin-similarity score
        csim = -w_coin * ((coins.unsqueeze(3) - cw) ** 2).sum(-1) / sigma2

        # Field-potential-difference score (averaged across scales)
        pdiff = -w_phi * (phi_agg.unsqueeze(-1) - pp).abs().mean(1)

        score = (csim + pdiff).clamp(-20, 20)
        attn = F.softmax(score, dim=-1)

        # Attention-weighted neighbour feature aggregation
        neighbor_features = (attn.unsqueeze(-1) * cw).sum(3)   # [B, H, W, K]
        return attn, neighbor_features


In [5]:
# === 2D Physics Feature Extractor ===

class FeatureExtractor2D(nn.Module):
    """Assemble composite feature vector from 2D field quantities."""
    def __init__(self, n_gen, n_scales=3, n_dirs=4, win_size=9):
        super().__init__()
        self.n_gen = n_gen
        self.n_scales = n_scales
        self.win_size = win_size
        # Feature composition:
        #   K (own coins) + K (neighbour aggregate) + win_size (attn weights)
        #   + n_dirs*n_scales (phi values) + n_dirs*n_scales (grad_phi values)
        #   + 1 (density) + 1 (attention entropy)
        self.feat_dim = (n_gen + n_gen + win_size
                         + n_dirs * n_scales + n_dirs * n_scales
                         + 1 + 1)

    def forward(self, coins, attn, neighbor_features, rho, phi, grad_phi):
        """
        coins:             [B, H, W, K]
        attn:              [B, H, W, win]
        neighbor_features: [B, H, W, K]
        rho:               [B, H, W]
        phi, grad_phi:     [B, n_dirs, S, H, W]
        """
        B, H, W, K = coins.shape
        feats = []

        # 1. Own coins
        feats.append(coins)

        # 2. Neighbour-aggregated coin features
        feats.append(neighbor_features)

        # 3. Local attention weights
        feats.append(attn)

        # 4. Phi values per direction per scale  (reshape from [B, ND, S, H, W])
        feats.append(phi.permute(0, 3, 4, 1, 2).reshape(B, H, W, -1))

        # 5. Grad_phi values per direction per scale
        feats.append(grad_phi.permute(0, 3, 4, 1, 2).reshape(B, H, W, -1))

        # 6. Density
        feats.append(rho.unsqueeze(-1))

        # 7. Attention entropy
        ac = attn.clamp(min=1e-15)
        feats.append(-(ac * ac.log()).sum(-1, keepdim=True))

        return torch.cat(feats, dim=-1)


# === Gravity 2D Block ===

class GravityBlock2D(nn.Module):
    """Single Gravity-2D block (single-channel C=1)."""
    def __init__(self, d_model, n_gen=15, n_scales=3, radius=1, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.coin_proj = nn.Linear(d_model, n_gen)

        self.field_solver = FieldSolver2D(n_scales)
        self.attention    = LocalAttention2D(n_gen, n_scales, radius)
        win = (2 * radius + 1) ** 2
        self.features    = FeatureExtractor2D(n_gen, n_scales, n_dirs=4, win_size=win)

        self.feat_proj = nn.Sequential(
            nn.Linear(self.features.feat_dim, d_model),
            nn.GELU(),
            nn.Dropout(dropout))

        # Gated residual connection (initialised to pass-through)
        self.gate = nn.Linear(d_model, d_model)
        nn.init.zeros_(self.gate.weight)
        nn.init.ones_(self.gate.bias)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout))

        nn.init.normal_(self.coin_proj.weight, 0, 0.02)
        nn.init.zeros_(self.coin_proj.bias)

    def forward(self, x):
        # x: [B, H, W, d_model]
        h = self.norm1(x)
        coins = math.pi * torch.tanh(self.coin_proj(h))   # [B,H,W,K]
        rho = self.attention.compute_density(coins)        # [B,H,W]
        phi, grad_phi = self.field_solver(rho)             # [B,4,S,H,W]
        attn, nf = self.attention(coins, phi)              # [B,H,W,win], [B,H,W,K]
        feats = self.features(coins, attn, nf, rho, phi, grad_phi)
        x = x + torch.sigmoid(self.gate(h)) * self.feat_proj(feats)
        x = x + self.ffn(self.norm2(x))
        return x


In [6]:
# === Gravity 2D model for CIFAR-10 classification ===

class Gravity2D(nn.Module):
    """Gravity-2D image classifier."""
    def __init__(self, n_classes=10, d_model=128, n_gen=15, n_scales=3,
                 radius=1, n_layers=4, patch_size=4, img_size=32, dropout=0.1):
        super().__init__()
        self.patch_size = patch_size
        self.grid_h = img_size // patch_size
        self.grid_w = img_size // patch_size

        # Patch embedding via linear projection of pixel patches
        self.patch_embed = nn.Linear(3 * patch_size * patch_size, d_model)
        self.pos_embed   = nn.Parameter(
            torch.randn(1, self.grid_h, self.grid_w, d_model) * 0.02)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            GravityBlock2D(d_model, n_gen, n_scales, radius, dropout)
            for _ in range(n_layers)])

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, imgs):
        # imgs: [B, 3, 32, 32]
        B = imgs.shape[0]
        P = self.patch_size
        # Patchify: unfold spatial dims into patches of size P x P
        x = imgs.unfold(2, P, P).unfold(3, P, P) \
                .permute(0, 2, 3, 1, 4, 5) \
                .reshape(B, self.grid_h, self.grid_w, -1)     # [B, gh, gw, 3*P*P]
        x = self.drop(self.patch_embed(x) + self.pos_embed)
        for block in self.blocks:
            x = block(x)
        # Global average pool over spatial dims, then classify
        return self.head(self.norm(x).mean(dim=(1, 2)))


## 3. Build model and verify parameter count

The exact parameter count is approximately **650K** for the default configuration
(d=128, K=15, S=3, R=1, 4 layers, 4×4 patches on a 32×32 image). The paper's
reported 653K corresponds to this configuration; minor implementation differences
in the feature projection layer may shift the count by a few thousand parameters.

The reproduction target is **test accuracy ≈ 82.3%**, which is robust to these
minor implementation variations.


In [7]:
# === Build model ===

D_MODEL    = 128
K          = 15
S          = 3
RADIUS     = 1       # → 3x3 window, win_size=9
N_LAYERS   = 4
PATCH_SIZE = 4       # 32/4 = 8x8 grid
IMG_SIZE   = 32
DROPOUT    = 0.1
N_CLASSES  = 10

model = Gravity2D(
    n_classes=N_CLASSES, d_model=D_MODEL, n_gen=K, n_scales=S,
    radius=RADIUS, n_layers=N_LAYERS, patch_size=PATCH_SIZE,
    img_size=IMG_SIZE, dropout=DROPOUT
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {n_params:,} ({n_params/1e3:.1f}K)")
print(f"Paper 1 Table 10 reports: ~653K\n")

# Sanity-check forward pass
with torch.no_grad():
    sample = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=device)
    out = model(sample)
print(f"Sanity forward: input {tuple(sample.shape)} -> output {tuple(out.shape)}")
print(f"Output finite: {torch.isfinite(out).all().item()}")


Total parameters: 652,654 (652.7K)
Paper 1 Table 10 reports: ~653K

Sanity forward: input (2, 3, 32, 32) -> output (2, 10)
Output finite: True


## 4. Training: AdamW + cosine schedule, 50 epochs

- Optimizer: AdamW (lr=2e-3, weight_decay=0.05)
- LR schedule: cosine decay over 50 epochs
- Gradient clipping: max_norm=1.0
- Mixed precision (AMP) when CUDA available
- Loss: cross-entropy over 10 classes


In [8]:
# === Training utilities ===

EPOCHS = 50
LR = 2e-3
WD = 0.05
GRAD_CLIP = 1.0

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD,
                              betas=(0.9, 0.95))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda') if use_amp else None

def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            with torch.amp.autocast('cuda'):
                logits = model(imgs)
                loss = F.cross_entropy(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(imgs)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total   += labels.size(0)
    return total_loss / total, 100.0 * correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        if use_amp:
            with torch.amp.autocast('cuda'):
                logits = model(imgs)
        else:
            logits = model(imgs)
        correct += (logits.argmax(1) == labels).sum().item()
        total   += labels.size(0)
    return 100.0 * correct / total


In [9]:
# === Training loop ===

history = {'train_loss': [], 'train_acc': [], 'test_acc': [], 'epoch_time': []}
best_test_acc = 0.0

print(f"Training Gravity-2D for {EPOCHS} epochs on {device}")
print(f"Total optimizer steps: {EPOCHS * len(train_loader)}")
print(f"Steps per epoch: {len(train_loader)}")
print("=" * 75)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scaler)
    test_acc = evaluate(model, test_loader)
    scheduler.step()
    epoch_time = time.time() - t0

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_acc'].append(test_acc)
    history['epoch_time'].append(epoch_time)

    marker = ""
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        marker = " ← best"

    if epoch <= 3 or epoch % 5 == 0 or marker:
        print(f"Epoch {epoch:3d}/{EPOCHS} | loss={train_loss:.4f} | "
              f"train_acc={train_acc:.2f}% | test_acc={test_acc:.2f}% | "
              f"time={epoch_time:.1f}s{marker}")

print("=" * 75)
print(f"Best test accuracy: {best_test_acc:.2f}%")
print(f"Paper target:        82.3%")


Training Gravity-2D for 50 epochs on cuda
Total optimizer steps: 19500
Steps per epoch: 390
Epoch   1/50 | loss=1.7858 | train_acc=33.73% | test_acc=44.30% | time=29.5s ← best
Epoch   2/50 | loss=1.4803 | train_acc=46.01% | test_acc=52.61% | time=29.0s ← best
Epoch   3/50 | loss=1.3194 | train_acc=52.15% | test_acc=55.37% | time=29.0s ← best
Epoch   4/50 | loss=1.2108 | train_acc=56.31% | test_acc=60.07% | time=29.0s ← best
Epoch   5/50 | loss=1.1378 | train_acc=59.30% | test_acc=62.80% | time=28.7s ← best
Epoch   6/50 | loss=1.0726 | train_acc=61.81% | test_acc=64.47% | time=29.0s ← best
Epoch   7/50 | loss=1.0338 | train_acc=63.22% | test_acc=65.49% | time=29.3s ← best
Epoch   8/50 | loss=0.9802 | train_acc=65.14% | test_acc=66.38% | time=29.6s ← best
Epoch   9/50 | loss=0.9412 | train_acc=66.50% | test_acc=68.21% | time=30.5s ← best
Epoch  10/50 | loss=0.9061 | train_acc=67.70% | test_acc=69.20% | time=29.7s ← best
Epoch  12/50 | loss=0.8616 | train_acc=69.51% | test_acc=71.36% | ti

## 5. Final test accuracy

Report the best test accuracy achieved during training:


In [10]:
# === Final result ===

final_test_acc = evaluate(model, test_loader)

print(f"Final test accuracy (last epoch): {final_test_acc:.2f}%")
print(f"Best test accuracy:                {best_test_acc:.2f}%")
print()
print(f"Paper 1 Table 10 target:           82.3%")
print(f"Acceptable reproduction range:     80.0% - 84.0%")

ok = 80.0 <= best_test_acc <= 84.0
print(f"\nReproduction status: {'PASS' if ok else 'INVESTIGATE'}")


Final test accuracy (last epoch): 81.81%
Best test accuracy:                81.81%

Paper 1 Table 10 target:           82.3%
Acceptable reproduction range:     80.0% - 84.0%

Reproduction status: PASS


## 6. Save checkpoint (optional)

Save model weights for verification or downstream use. The expected SHA-256
hash is documented in `checkpoints/README.md` of the gravity-nn repository.


In [11]:
# === Save checkpoint ===

ckpt_path = 'gravity_2d_cifar10_653k.pt'

state = {
    'model_state_dict': model.state_dict(),
    'n_classes': N_CLASSES,
    'config': {
        'd_model': D_MODEL, 'n_gen': K, 'n_scales': S, 'radius': RADIUS,
        'n_layers': N_LAYERS, 'patch_size': PATCH_SIZE, 'img_size': IMG_SIZE,
        'dropout': DROPOUT,
        'batch_size': BATCH_SIZE, 'epochs': EPOCHS,
        'lr': LR, 'weight_decay': WD, 'seed': SEED,
    },
    'metrics': {
        'best_test_acc': best_test_acc,
        'final_test_acc': final_test_acc,
    },
    'history': history,
}
torch.save(state, ckpt_path)

# Hash the checkpoint for reproducibility
sha = hashlib.sha256()
with open(ckpt_path, 'rb') as f:
    for chunk in iter(lambda: f.read(8192), b''):
        sha.update(chunk)

print(f"Checkpoint saved: {ckpt_path}")
print(f"Size: {os.path.getsize(ckpt_path) / 1024:.1f} KB")
print(f"SHA-256: {sha.hexdigest()}")


Checkpoint saved: gravity_2d_cifar10_653k.pt
Size: 2583.0 KB
SHA-256: 832c387241d110df7000ada291bf83fcd80f2b18f55cbf2ad27ffa57416c03ff


## References

If you use this notebook or the Gravity architecture in your research, please cite:

```bibtex
@article{chiang2026gravity,
  title={Gravity: A Physics-Inspired O(N) Framework with O(1) Streaming State Across Dimensions},
  author={Chiang, Chia-Wei},
  year={2026},
  publisher={Zenodo},
  doi={10.5281/zenodo.XXXXXXX}
}
```

## License

This notebook is licensed under AGPL-3.0. The Gravity architecture implements
technology covered by pending U.S. patent application (March 2026). Commercial
licensing inquiries: chiangjw90@gmail.com

## Repository

Full reference implementation: https://github.com/chiangjw90/gravity-nn

## Companion notebooks

- `gravity_1d_wikitext2.ipynb` — Paper 1 Table 2 (PPL 4.36)
- **`gravity_2d_cifar10.ipynb`** — Paper 1 Table 10 (82.3% accuracy) — this notebook
- `gravity_3d_organmnist.ipynb` — Paper 1 Table 11 (89.7% accuracy)
